In [ ]:
import psycopg2
import pandas as pd
from collections import namedtuple
db_host = "localhost"
db_name = "Franchise"
db_user = "postgres"
db_password = "admin"
db_port = "5433"

try:
    connection = psycopg2.connect(host=db_host, database=db_name, user=db_user, password=db_password, port=db_port)
    print("Connexion à la base de données réussie")
except psycopg2.Error as e:
    print("Échec de la connexion à la base de données:", e)


Connexion à la base de données réussie


In [ ]:
data = pd.read_excel(r"C:\Users\User\restaurants\data_VF.xlsx")
print(data.head())


             nom  apport_personnel  droites_entree  investissement_global  \
0  YATTA ! RAMEN             80000           30000                 300000   
1         WOODIZ             30000           45000                 300000   
2        WENDY'S            300000           33000                 300000   
3    TUTTI PIZZA             70000           15000                 120000   
4     THE ROSTER             70000           50000                 300000   

   CA_potentielApres2ans  Implantations Formations Aide_auFinancement  \
0                 650000              4        oui                oui   
1                 650000             62        oui                oui   
2                 500000           7000        non                non   
3                 650000             75        oui                oui   
4                1000000             10        oui                non   

  Redevance_fonctionnement Redevance_publicitaire  ...  \
0                       5%              

In [ ]:
Dimension = namedtuple('Dimension', ['name', 'key', 'attributes'])
FactTable = namedtuple('FactTable', ['name', 'keyrefs', 'measures'])

# Dimensions
franchise_info_dimension = Dimension(
    name='franchise_info',
    key='id_franchise',
    attributes=['nom', 'Type_de_contrat', 'Surface_moyenne', 'Formations', 'Aide_auFinancement', 'CompanyStores'],

)
annee_dimension = Dimension(
    name='annee',
    key='id_annee',
    attributes=['date_creation']
)
heure_dimension = Dimension(
    name='heure',
    key='id_heure',
    attributes=['heure_ouverture', 'heure_fermeture']
)
adresse_dimension = Dimension(
    name='adresse',
    key='id_adresse',
    attributes=['adresses']
)
avis_dimension = Dimension(
    name='avis',
    key='id_avis',
    attributes=['avis', 'rating', 'food_rating', 'service_rating']
)
categorie_dimension = Dimension(
    name='categorie',
    key='id_categorie',
    attributes=['Merchant_Category']
)

cuisine_dimension = Dimension(
    name='cuisine',
    key='id_cuisine',
    attributes=['cuisines']
)

redevance_dimension = Dimension(
    name='redevance',
    key='id_redevance',
    attributes=['Redevance_fonctionnement', 'Redevance_publicitaire']
)

enabled_options_dimension = Dimension(
    name='enabled_options',
    key='id_option',
    attributes=['pickup_enabled', 'delivery_enabled']
)



# Table de fait
franchise_Fact = FactTable(
    name='Franchise',
    keyrefs=['id_franchise', 'id_annee', 'id_heure', 'id_adresse', 'id_avis', 'id_categorie', 'id_cuisine', 'id_redevance', 'id_option'],
    measures=['CA_potentielApres2ans', 'Implantations', 'company_owned_units_2019', 'total_units_2019', 'sales_in_millions_2019', 'unit_change_from_2018', 'YOY_Sales', 'YOY_Units', 'TotalUnits2021', 'TotalChange_in_Units_from2020', 'apport_personnel', 'droites_entree', 'investissement_global', 'franchise_fee']
)


In [ ]:
cursor = connection.cursor()
for index, row in data.iterrows():
    nom = row['nom']
    type_contrat = row['Type_de_contrat']
    surface_moyenne = row['Surface_moyenne']
    formations = row['Formations']
    aide_financement = row['Aide_auFinancement']
    company_stores = row['CompanyStores']


    cursor.execute("INSERT INTO franchise_info (nom, Type_de_contrat, Surface_moyenne, Formations, Aide_auFinancement, CompanyStores) VALUES (%s, %s, %s, %s, %s, %s) RETURNING id_franchise", (nom, type_contrat, surface_moyenne, formations, aide_financement, company_stores))
    franchise_info_id = cursor.fetchone()[0]
    connection.commit()


In [ ]:
cursor = connection.cursor()
for index, row in data.iterrows():
    date_creation = row['date_creation']

    cursor.execute("INSERT INTO annee (date_creation) VALUES (%s) RETURNING id_annee", (date_creation,))
    annee_id = cursor.fetchone()[0]
    connection.commit()


In [ ]:
cursor = connection.cursor()
for index, row in data.iterrows():
    heure_ouverture = row['heure_ouverture']
    heure_fermeture = row['heure_fermeture']


    cursor.execute("INSERT INTO heure (heure_ouverture, heure_fermeture) VALUES (%s, %s) RETURNING id_heure", (heure_ouverture, heure_fermeture))
    heure_id = cursor.fetchone()[0]
    connection.commit()


In [ ]:
cursor = connection.cursor()
for index, row in data.iterrows():
    adresse = row['adresses']

    cursor.execute("INSERT INTO adresse (adresses) VALUES (%s) RETURNING id_adresse", (adresse,))
    adresse_id = cursor.fetchone()[0]
    connection.commit()


In [ ]:
cursor = connection.cursor()
for index, row in data.iterrows():
    avis = row['avis']
    rating = row['rating']
    food_rating = row['food_rating']
    service_rating = row['service_rating']

    cursor.execute("INSERT INTO avis (avis, rating, food_rating, service_rating) VALUES (%s, %s, %s, %s) RETURNING id_avis", (avis, rating, food_rating, service_rating))
    avis_id = cursor.fetchone()[0]
    connection.commit()


In [ ]:
cursor = connection.cursor()
for index, row in data.iterrows():
    merchant_category = row['Merchant_Category']

    cursor.execute("INSERT INTO categorie (Merchant_Category) VALUES (%s) RETURNING id_categorie", (merchant_category,))
    categorie_id = cursor.fetchone()[0]
    connection.commit()



In [ ]:
cursor = connection.cursor()
for index, row in data.iterrows():
    cuisines = row['cuisines']

    cursor.execute("INSERT INTO cuisine (cuisines) VALUES (%s) RETURNING id_cuisine", (cuisines,))
    cuisine_id = cursor.fetchone()[0]
    connection.commit()


In [ ]:
cursor = connection.cursor()
for index, row in data.iterrows():
    redev_fonctionnement = row['Redevance_fonctionnement']
    redev_publicitaire = row['Redevance_publicitaire']

    cursor.execute("INSERT INTO redevance (Redevance_fonctionnement, Redevance_publicitaire) VALUES (%s, %s) RETURNING id_redevance", (redev_fonctionnement, redev_publicitaire))
    redevance_id = cursor.fetchone()[0]
    connection.commit()


In [ ]:
cursor = connection.cursor()
for index, row in data.iterrows():
    pickup_enabled = row['pickup_enabled']
    delivery_enabled = row['delivery_enabled']

    cursor.execute("INSERT INTO enabled_options (pickup_enabled, delivery_enabled) VALUES (%s, %s) RETURNING id_option", (pickup_enabled, delivery_enabled))
    option_id = cursor.fetchone()[0]
    connection.commit()


In [ ]:
cursor = connection.cursor()
for index, row in data.iterrows():
    # Récupérer les clés étrangères correspondantes à partir des tables de dimensions
    cursor.execute("SELECT id_franchise FROM franchise_info WHERE nom = %s", (row['nom'],))
    id_franchise = cursor.fetchone()[0]

    cursor.execute("SELECT id_annee FROM annee WHERE date_creation = %s", (row['date_creation'],))
    id_annee = cursor.fetchone()[0]

    cursor.execute("SELECT id_heure FROM heure WHERE heure_ouverture = %s AND heure_fermeture = %s", (row['heure_ouverture'], row['heure_fermeture']))
    id_heure = cursor.fetchone()[0]

    cursor.execute("SELECT id_adresse FROM adresse WHERE adresses = %s", (row['adresses'],))
    id_adresse = cursor.fetchone()[0]

    cursor.execute("SELECT id_avis FROM avis WHERE avis = %s AND rating = %s AND food_rating = %s AND service_rating = %s", (row['avis'], row['rating'], row['food_rating'], row['service_rating']))
    id_avis = cursor.fetchone()[0]

    cursor.execute("SELECT id_categorie FROM categorie WHERE Merchant_Category = %s", (row['Merchant_Category'],))
    id_categorie = cursor.fetchone()[0]

    cursor.execute("SELECT id_cuisine FROM cuisine WHERE cuisines = %s", (row['cuisines'],))
    id_cuisine = cursor.fetchone()[0]

    cursor.execute("SELECT id_redevance FROM redevance WHERE Redevance_fonctionnement = %s AND Redevance_publicitaire = %s", (row['Redevance_fonctionnement'], row['Redevance_publicitaire']))
    id_redevance = cursor.fetchone()[0]

    cursor.execute("SELECT id_option FROM enabled_options WHERE pickup_enabled = %s AND delivery_enabled = %s", (row['pickup_enabled'], row['delivery_enabled']))
    id_option = cursor.fetchone()[0]

    # Insérer les données dans la table de faits avec les clés étrangères
    cursor.execute("INSERT INTO franchise (id_franchise, id_annee, id_heure, id_adresse, id_avis, id_categorie, id_cuisine, id_redevance, id_option, CA_potentielApres2ans, Implantations, company_owned_units_2019, total_units_2019, sales_in_millions_2019, unit_change_from_2018, YOY_Sales, YOY_Units, TotalUnits2021, TotalChange_in_Units_from2020, apport_personnel, droites_entree, investissement_global, franchise_fee) VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s)",
                   (id_franchise, id_annee, id_heure, id_adresse, id_avis, id_categorie, id_cuisine, id_redevance, id_option, row['CA_potentielApres2ans'], row['Implantations'], row['company_owned_units_2019'], row['total_units_2019'], row['sales_in_millions_2019'], row['unit_change_from_2018'], row['YOY_Sales'], row['YOY_Units'], row['TotalUnits2021'], row['TotalChange_in_Units_from2020'], row['apport_personnel'], row['droites_entree'], row['investissement_global'], row['franchise_fee']))
    connection.commit()


In [ ]:
connection.close()
